# Seed Coding Books into MongoDB with Ollama Embeddings

This notebook seeds one or more books into MongoDB for RAG.

## What it does
- Reads `.txt`, `.md`, and optionally `.pdf` files from a folder
- Splits each book into overlapping word-count chunks
- Generates embeddings from a local Ollama model using `POST /api/embed`
- Stores each chunk plus metadata and the embedding in MongoDB
- Uses upserts so you can rerun safely without creating duplicates

## Collection shape
Each chunk document stored in `book_chunks` includes:
- `book_id` — slugified title used as stable compound-key prefix
- `title` — human-readable book title (file stem)
- `author` — configurable metadata
- `chunk_index` — sequential position within the book
- `text` — raw chunk text
- `embedding` — dense float vector from Ollama (`nomic-embed-text`, 768-dim)
- `tags` — free-form labels
- `source_type` — `"book"` by default
- `source` — human-readable provenance string, e.g. `"book/effective-java"` (used by the RAG pipeline's `context_to_text()`)
- `domain` — optional intent domain tag (`"java"`, `"python"`, `"sql"`, `"mongodb"`) used by the reranker's domain bonus

## Assumptions
- MongoDB is running and reachable at `MONGO_URI`
- Ollama is running locally with `nomic-embed-text` already pulled (`ollama pull nomic-embed-text`)
- Books are plain-text, markdown, or PDF files in `BOOKS_DIR`

> **Tip:** If your books are PDFs, install `pypdf` (see the cell below) for automatic extraction.


## Notes

MongoDB stores vector embeddings alongside document data. The `embedding` field holds a flat
array of floats per chunk. Ollama's `/api/embed` endpoint accepts a single string or an array
of strings and returns an `embeddings` array. PyMongo's `insert_one()` and `insert_many()` are
the standard insert operations for MongoDB documents.

In [2]:
# If needed, uncomment and run this cell once to install dependencies.
# %pip install pymongo requests pypdf

In [1]:
from pathlib import Path
from typing import Iterable, List, Dict, Optional
import hashlib
import re
import requests
from pymongo import MongoClient, ReplaceOne
from pymongo.errors import PyMongoError

try:
    from pypdf import PdfReader
    PDF_ENABLED = True
except Exception:
    PDF_ENABLED = False

## Configuration

Update these values to match your local environment.

In [2]:
# --- MongoDB ---
MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "engineering_copilot"
COLLECTION_NAME = "book_chunks"

# --- Ollama embeddings ---
OLLAMA_EMBED_URL = "http://localhost:11434/api/embed"
EMBED_MODEL = "nomic-embed-text"
EMBED_BATCH_SIZE = 16

# --- Input books folder ---
BOOKS_DIR = Path("../books")

# --- Chunking ---
# nomic-embed-text is commonly documented with a larger context window,
# but effective limits can vary by Ollama build/model metadata. Keep chunk
# sizing moderate and rely on the hard char cap below before each embed call.
CHUNK_WORDS = 500
CHUNK_OVERLAP_WORDS = 75

# --- Embedding safety guard ---
# Hard character cap applied before every Ollama embed call.
# This is intentionally conservative to avoid "input length exceeds context
# length" across different Ollama/model combinations.
MAX_CHUNK_CHARS = 1200

# --- Token-length filter for split_words ---
# Any whitespace-delimited token longer than this is almost certainly garbage
# from PDF extraction (base64 image data, binary streams, etc.) and is dropped.
MAX_WORD_CHARS = 200

# --- Metadata defaults ---
DEFAULT_AUTHOR = "Unknown"
DEFAULT_TAGS = ["book", "coding"]

# --- Source type tag ---
SOURCE_TYPE = "book"

# --- Book → domain mapping ---
# Keys are the slugified file stem (what slugify(file.stem) produces).
# The domain drives TWO behaviours in the agent:
#   1. Reranker domain bonus  — _domain_bonus() in rag.py boosts chunk scores
#      when the detected question domain matches the chunk domain.
#   2. Prompt routing         — agent_runtime.py selects a specialised prompt
#      (e.g. code_deep_learning, code_algorithms) based on the detected domain.
#
# Valid domain values and their meaning:
#   "java"          — Java development (Spring Boot, JVM, Maven)
#   "deep_learning" — PyTorch, TensorFlow, Keras, neural architectures
#   "data_science"  — Pandas, NumPy, Matplotlib, Seaborn, scikit-learn
#   "algorithms"    — Data structures, sorting, graphs, Big-O analysis
#   "security"      — Network security, cryptography, penetration testing
#   "finance"       — Quantitative finance, trading, portfolio management
#   "ml"            — ML theory: Bayesian stats, linear algebra, probability
#   "platform"      — DevOps, Kubernetes, cloud infrastructure, CI/CD
#   "python"        — General Python (core language, expert usage)
#   None            — No domain bonus; falls back to qa/code_python prompts
BOOK_DOMAIN_MAP: dict = {
    # Java
    "java-developer-guide": "java",
    "java-a-beginner-s-guide-9th-edition": "java",
    "spring-in-action": "java",

    # Deep learning
    "deep-learning-with-pytorch": "deep_learning",
    "t0-deep-learning-with-pytorch-packt2018-book": "deep_learning",
    "deep-learning-with-tensorflow-20-and-keras": "deep_learning",

    # Data science
    "pandas-cookbook-ebook": "data_science",

    # Algorithms and data structures
    "python-algorithms": "algorithms",
    "grokking-algorithms-2nd-edition-meap": "algorithms",

    # Security
    "black-hat-python-pdfdrive": "security",

    # Quantitative finance
    "mastering-python-for-finance": "finance",

    # Machine learning theory / math
    "ai-foundations-of-computational-agents-3rd-ed": "ml",
    "thinkbayes": "ml",
    "linear-algebra": "ml",

    # Platform / DevOps
    "platform-engineering": "platform",
    "awsmp-atton1u6cf2e6pglv-awsmp-gim-kmpc-adhoc-mpd-devops-ai-hbr-devops-whitepaper": "platform",

    # General Python (core language)
    "expert-python-programming": "python",
    "thinkpython2": "python",
}


## Helpers: File Loading and Chunking

In [3]:
def slugify(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "-", text)
    return text.strip("-")


def read_text_file(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")


def read_pdf_file(path: Path) -> str:
    if not PDF_ENABLED:
        raise RuntimeError("pypdf is not installed. Run: %pip install pypdf")
    reader = PdfReader(str(path))
    pages = []
    for page in reader.pages:
        pages.append(page.extract_text() or "")
    return "\n\n".join(pages)


def load_book_text(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in {".txt", ".md"}:
        return read_text_file(path)
    if suffix == ".pdf":
        return read_pdf_file(path)
    raise ValueError(f"Unsupported file type: {path.suffix}")


In [4]:
def split_words(text: str) -> List[str]:
    """Split text on whitespace, dropping tokens that are clearly garbage
    from PDF extraction (base64 blobs, binary streams, very long URLs, etc.).
    Tokens longer than MAX_WORD_CHARS are silently discarded — they would
    produce thousands of embedding tokens and trigger the Ollama context-length
    error without contributing useful semantic signal.
    """
    return [w for w in re.findall(r"\S+", text) if len(w) <= MAX_WORD_CHARS]


def chunk_text(text: str, chunk_words: int = 500, overlap_words: int = 75) -> List[str]:
    words = split_words(text)
    if not words:
        return []

    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_words, len(words))
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)

        if end == len(words):
            break

        start = max(end - overlap_words, 0)

    return chunks


## Helpers: Embeddings with Ollama

Ollama's `/api/embed` endpoint accepts either a single string or an array of strings and
returns an `embeddings` array. This notebook batches multiple chunks per request for better
throughput.

In [5]:
def batched(items: List[str], size: int) -> Iterable[List[str]]:
    for i in range(0, len(items), size):
        yield items[i:i + size]


def _post_embed(payload: Dict) -> List[List[float]]:
    resp = requests.post(OLLAMA_EMBED_URL, json=payload, timeout=300)
    if resp.status_code >= 400:
        # Include the response body so the notebook shows the real Ollama error.
        raise requests.HTTPError(
            f"{resp.status_code} {resp.reason}: {resp.text}",
            response=resp,
        )
    data = resp.json()
    return data["embeddings"]


def embed_batch(texts: List[str], model: str = EMBED_MODEL) -> List[List[float]]:
    # Truncate each text to MAX_CHUNK_CHARS as a hard safety net before the
    # Ollama API call. split_words already filters garbage tokens, but this
    # guard also protects against any chunk that slipped through.
    safe_texts = [t[:MAX_CHUNK_CHARS] for t in texts]

    # Fast path: send the full batch in one call.
    try:
        return _post_embed({"model": model, "input": safe_texts})
    except requests.HTTPError as exc:
        msg = str(exc).lower()
        # Compatibility fallback: some Ollama versions effectively enforce
        # context limits across a batch request. If that happens, retry each
        # text individually so one oversized aggregate request cannot fail
        # the whole batch.
        if "context length" not in msg and "input length" not in msg:
            raise

    embeddings: List[List[float]] = []
    for i, text in enumerate(safe_texts):
        try:
            one = _post_embed({"model": model, "input": text})
            embeddings.append(one[0])
        except requests.HTTPError as single_exc:
            print(f"  Embed failed for item {i} (chars={len(text)}): {single_exc}")
            raise
    return embeddings


def embed_texts(texts: List[str], batch_size: int = EMBED_BATCH_SIZE) -> List[List[float]]:
    all_embeddings: List[List[float]] = []
    for batch in batched(texts, batch_size):
        all_embeddings.extend(embed_batch(batch))
    return all_embeddings


## MongoDB Setup

Creates the MongoDB client, selects the target collection, and ensures a unique compound
index on `(book_id, chunk_index)` so upserts are idempotent.

In [6]:
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

# Compound unique index: one chunk per (book, position) — ensures safe upserts.
collection.create_index(
    [("book_id", 1), ("chunk_index", 1)],
    unique=True,
    name="book_chunk_idx",
)

# Index on source_type to support efficient retrieval queries.
collection.create_index(
    [("source_type", 1)],
    name="source_type_idx",
)

# NOTE: If you changed the embedding model (e.g. from embeddinggemma to nomic-embed-text),
# all stored embeddings are in a different vector space. Re-run this notebook to
# regenerate embeddings, then call POST /admin/reindex on the python-agent to
# re-embed the internal chunks collection as well.

print(f"Connected to {MONGO_URI}  |  db={DB_NAME}  |  collection={COLLECTION_NAME}")


Connected to mongodb://localhost:27017  |  db=engineering_copilot  |  collection=book_chunks


## Build Chunk Documents

`build_chunk_docs` takes a single file, chunks it, embeds every chunk, and returns a list of
MongoDB documents ready for upsert. The upsert key is the compound `(book_id, chunk_index)`
pair, so reruns are safe.

The `source` field (`"book/<book_id>"`) is required by the RAG pipeline's `context_to_text()`
function in `rag.py`, which renders `Source: <value>` in the LLM prompt. Without it, retrieved
book chunks would show `Source: None`.

The `domain` field is optional but used by the reranker's `_domain_bonus()` to boost results
that match the detected intent domain (e.g. `"java"`, `"python"`, `"sql"`, `"mongodb"`).


In [7]:
def build_chunk_docs(
    file_path: Path,
    author: str = DEFAULT_AUTHOR,
    tags: Optional[List[str]] = None,
    domain: Optional[str] = None,
    chunk_words: int = CHUNK_WORDS,
    overlap_words: int = CHUNK_OVERLAP_WORDS,
) -> List[Dict]:
    title = file_path.stem
    book_id = slugify(title)
    tags = tags or DEFAULT_TAGS

    text = load_book_text(file_path)
    chunks = chunk_text(text, chunk_words=chunk_words, overlap_words=overlap_words)
    embeddings = embed_texts(chunks)

    docs = []
    for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        doc = {
            "book_id": book_id,
            "title": title,
            "author": author,
            "chunk_index": i,
            "text": chunk,
            "embedding": embedding,
            "tags": tags,
            "source_type": SOURCE_TYPE,
            # `source` matches the field read by context_to_text() in rag.py
            "source": f"book/{book_id}",
            # `domain` used by the reranker domain bonus; None is valid (no boost)
            "domain": domain,
        }
        docs.append(doc)

    print(f"  {title}: {len(chunks)} chunks")
    return docs


## Upsert Chunk Documents into MongoDB

Uses PyMongo's `bulk_write` with `ReplaceOne(..., upsert=True)` on the compound key
`(book_id, chunk_index)`. This means reruns update existing documents rather than
inserting duplicates.

In [8]:
def upsert_chunk_docs(docs: List[Dict]) -> None:
    if not docs:
        return

    operations = [
        ReplaceOne(
            filter={"book_id": d["book_id"], "chunk_index": d["chunk_index"]},
            replacement=d,
            upsert=True,
        )
        for d in docs
    ]

    try:
        result = collection.bulk_write(operations, ordered=False)
        print(
            f"  Upserted {result.upserted_count}, "
            f"modified {result.modified_count} documents."
        )
    except PyMongoError as exc:
        print(f"  MongoDB error: {exc}")

## Seed All Books from a Folder

`seed_books_from_folder` iterates every supported file in `BOOKS_DIR`, looks up each file's
domain from `BOOK_DOMAIN_MAP` using its slugified stem, builds chunk documents, and upserts
them. Supported extensions: `.txt`, `.md`, and (if `pypdf` is installed) `.pdf`.

### Domain → prompt routing

| Domain | Prompt selected | Books |
|---|---|---|
| `java` | `code_java` | java_developer_guide, Java Beginner's Guide, Spring in Action |
| `deep_learning` | `code_deep_learning` | PyTorch ×2, TF+Keras |
| `data_science` | `code_data_science` | Pandas Cookbook |
| `algorithms` | `code_algorithms` | Python Algorithms, Grokking Algorithms |
| `security` | `code_security` | Black Hat Python |
| `finance` | `code_finance` | Mastering Python for Finance |
| `ml` | `qa` + ML reranker boost | AI Foundations, Think Bayes, Linear Algebra |
| `platform` | `qa` + platform reranker boost | Platform Engineering, AWS whitepaper |
| `python` | `code_python` | Expert Python, Think Python 2 |
| `None` | `qa` / `code_python` (no boost) | — |

To override a book's domain or add a new book, call `build_chunk_docs` directly:

```python
docs = build_chunk_docs(Path("../books/my-book.pdf"), domain="java")
upsert_chunk_docs(docs)
```


In [9]:
SUPPORTED_SUFFIXES = {".txt", ".md"}
if PDF_ENABLED:
    SUPPORTED_SUFFIXES.add(".pdf")


def seed_books_from_folder(folder: Path) -> None:
    if not folder.exists():
        print(f"Folder not found: {folder.resolve()}")
        return

    files = sorted(
        f for f in folder.iterdir()
        if f.is_file() and f.suffix.lower() in SUPPORTED_SUFFIXES
    )

    if not files:
        print(f"No supported files in {folder.resolve()}")
        return

    print(f"Seeding {len(files)} book(s) from {folder.resolve()}\n")

    for file_path in files:
        import re as _re
        _slug = _re.sub(r"[^a-z0-9]+", "-", file_path.stem.lower().strip()).strip("-")
        domain = BOOK_DOMAIN_MAP.get(_slug)
        domain_note = f"  [domain={domain}]" if domain else "  [domain=None — no reranker bonus]"
        print(f"Processing: {file_path.name}{domain_note}")
        try:
            docs = build_chunk_docs(file_path, domain=domain)
            upsert_chunk_docs(docs)
        except Exception as exc:
            print(f"  ERROR processing {file_path.name}: {exc}")

    print("\nDone.")


## Run the Seeding Job

Run this cell to process all books in `BOOKS_DIR`.  
Place your `.txt`, `.md`, or `.pdf` files in that folder first.

In [10]:
seed_books_from_folder(BOOKS_DIR)

Seeding 18 book(s) from C:\Users\mario\OneDrive\Desktop\ai_ml_expert\books

Processing: AI Foundations of Computational Agents 3rd Ed.pdf  [domain=ml]
  AI Foundations of Computational Agents 3rd Ed: 807 chunks
  Upserted 807, modified 0 documents.
Processing: awsmp-attOn1u6CF2E6Pglv-awsmp-gim-kmpc-adhoc-mpd-devops-ai-hbr-devops-whitepaper.pdf  [domain=platform]
  awsmp-attOn1u6CF2E6Pglv-awsmp-gim-kmpc-adhoc-mpd-devops-ai-hbr-devops-whitepaper: 12 chunks
  Upserted 12, modified 0 documents.
Processing: Black Hat Python ( PDFDrive ).pdf  [domain=security]
  Black Hat Python ( PDFDrive ): 116 chunks
  Upserted 116, modified 0 documents.
Processing: Deep-Learning-with-PyTorch.pdf  [domain=deep_learning]
  Deep-Learning-with-PyTorch: 424 chunks
  Upserted 424, modified 0 documents.
Processing: deep-learning-with-tensorflow-20-and-keras.pdf  [domain=deep_learning]
  deep-learning-with-tensorflow-20-and-keras: 338 chunks
  Upserted 338, modified 0 documents.
Processing: Expert_Python_Program

Ignoring wrong pointing object 2782 0 (offset 0)
Ignoring wrong pointing object 2783 0 (offset 0)
Ignoring wrong pointing object 4065 0 (offset 0)
Ignoring wrong pointing object 4066 0 (offset 0)
Ignoring wrong pointing object 4067 0 (offset 0)
Ignoring wrong pointing object 4068 0 (offset 0)
Ignoring wrong pointing object 4069 0 (offset 0)
Ignoring wrong pointing object 4070 0 (offset 0)
Ignoring wrong pointing object 4071 0 (offset 0)
Ignoring wrong pointing object 4072 0 (offset 0)
Ignoring wrong pointing object 4073 0 (offset 0)
Ignoring wrong pointing object 4074 0 (offset 0)
Ignoring wrong pointing object 4075 0 (offset 0)
Ignoring wrong pointing object 4076 0 (offset 0)
Ignoring wrong pointing object 4077 0 (offset 0)
Ignoring wrong pointing object 4078 0 (offset 0)
Ignoring wrong pointing object 4079 0 (offset 0)
Ignoring wrong pointing object 4080 0 (offset 0)
Ignoring wrong pointing object 4081 0 (offset 0)
Ignoring wrong pointing object 4082 0 (offset 0)
Ignoring wrong point

  Upserted 443, modified 0 documents.
Processing: mastering-python-for-finance.pdf  [domain=finance]


Ignoring wrong pointing object 4404 0 (offset 0)
Ignoring wrong pointing object 4405 0 (offset 0)
Ignoring wrong pointing object 4406 0 (offset 0)
Ignoring wrong pointing object 4407 0 (offset 0)
Ignoring wrong pointing object 4408 0 (offset 0)
Ignoring wrong pointing object 4409 0 (offset 0)
Ignoring wrong pointing object 4410 0 (offset 0)
Ignoring wrong pointing object 4411 0 (offset 0)
Ignoring wrong pointing object 4412 0 (offset 0)
Ignoring wrong pointing object 4413 0 (offset 0)
Ignoring wrong pointing object 4414 0 (offset 0)
Ignoring wrong pointing object 4415 0 (offset 0)
Ignoring wrong pointing object 4416 0 (offset 0)
Ignoring wrong pointing object 4417 0 (offset 0)
Ignoring wrong pointing object 4418 0 (offset 0)
Ignoring wrong pointing object 4419 0 (offset 0)
Ignoring wrong pointing object 4420 0 (offset 0)
Ignoring wrong pointing object 4421 0 (offset 0)
Ignoring wrong pointing object 4422 0 (offset 0)
Ignoring wrong pointing object 4423 0 (offset 0)
Ignoring wrong point

  mastering-python-for-finance: 172 chunks
  Upserted 172, modified 0 documents.
Processing: Pandas-Cookbook-eBook.pdf  [domain=data_science]
  Pandas-Cookbook-eBook: 231 chunks
  Upserted 231, modified 0 documents.
Processing: Platform_Engineering.pdf  [domain=platform]
  Platform_Engineering: 296 chunks
  Upserted 296, modified 0 documents.
Processing: Python.Algorithms.pdf  [domain=algorithms]
  Python.Algorithms: 393 chunks
  Upserted 393, modified 0 documents.
Processing: Spring in Action.pdf  [domain=java]
  Spring in Action: 347 chunks
  Upserted 347, modified 0 documents.
Processing: T0-deep learning with pytorch packt2018(book).pdf  [domain=deep_learning]
  T0-deep learning with pytorch packt2018(book): 126 chunks
  Upserted 126, modified 0 documents.
Processing: thinkbayes.pdf  [domain=ml]
  thinkbayes: 126 chunks
  Upserted 126, modified 0 documents.
Processing: thinkpython2.pdf  [domain=python]
  thinkpython2: 200 chunks
  Upserted 200, modified 0 documents.

Done.


## Inspect What Was Inserted

Run a quick sanity check — prints the first three chunks and the total count
stored in the collection.

In [11]:
total = collection.count_documents({})
print(f"Total documents in '{COLLECTION_NAME}': {total}\n")

# Spot-check: verify all required RAG fields are present
sample = list(collection.find(
    {},
    {"_id": 0, "book_id": 1, "chunk_index": 1, "title": 1,
     "source": 1, "source_type": 1, "domain": 1, "text": 1,
     "embedding": {"$slice": 3}},
).limit(3))

for doc in sample:
    emb_dim_note = f"embedding[0:3]={doc.get('embedding', [])!r}"
    print(f"book_id={doc['book_id']}  chunk_index={doc['chunk_index']}")
    print(f"  title       : {doc.get('title')!r}")
    print(f"  source      : {doc.get('source')!r}")
    print(f"  source_type : {doc.get('source_type')!r}")
    print(f"  domain      : {doc.get('domain')!r}")
    print(f"  {emb_dim_note}")
    print(f"  text preview: {doc['text'][:120]!r}")
    print()

# Warn if any docs are missing the source field (pre-existing docs from old runs)
missing_source = collection.count_documents({"source": {"$exists": False}})
if missing_source:
    print(f"WARNING: {missing_source} document(s) are missing the 'source' field.")
    print("Re-run seed_books_from_folder() to upsert updated docs with the source field.")
else:
    print("OK: all documents have the 'source' field.")


Total documents in 'book_chunks': 3954

book_id=ai-foundations-of-computational-agents-3rd-ed  chunk_index=0
  title       : 'AI Foundations of Computational Agents 3rd Ed'
  source      : 'book/ai-foundations-of-computational-agents-3rd-ed'
  source_type : 'book'
  domain      : None
  embedding[0:3]=[-0.013849102, 0.070561156, -0.1539016]
  text preview: 'Artiﬁcial Intelligence Foundations of Computational Agents Third Edition A comprehensive textbook for undergraduate and '

book_id=ai-foundations-of-computational-agents-3rd-ed  chunk_index=1
  title       : 'AI Foundations of Computational Agents 3rd Ed'
  source      : 'book/ai-foundations-of-computational-agents-3rd-ed'
  source_type : 'book'
  domain      : None
  embedding[0:3]=[-0.021366686, 0.06433194, -0.17511375]
  text preview: 'in which ﬁrst-order logic and relational AI are covered later, after thoroughly covering more basic, feature-based metho'

book_id=ai-foundations-of-computational-agents-3rd-ed  chunk_index=2
  tit

## Optional: Query-Time Embedding

Use `embed_query` to embed a search string and retrieve the most similar chunks
using a cosine similarity scan. This is useful for ad-hoc testing before wiring
full vector search.

In [ ]:
def embed_query(text: str) -> List[float]:
    return embed_batch([text])[0]


# Example usage
query_vector = embed_query("how does garbage collection work in Java?")
print(f"Query embedding dimension: {len(query_vector)}")
print(f"First 5 values: {query_vector[:5]}")

## Optional: MongoDB Atlas Vector Search

If you migrate to **MongoDB Atlas**, you can create a Vector Search index and run
`$vectorSearch` aggregation pipelines for ANN retrieval. The cell below shows the pattern
— uncomment and adapt once you have an Atlas cluster with a vector search index configured.

In [ ]:
# Atlas Vector Search example — requires an Atlas cluster with a Vector Search index
# named "embedding_index" on the "embedding" field.
#
# query_vec = embed_query("What is the observer pattern?")
#
# pipeline = [
#     {
#         "$vectorSearch": {
#             "index": "embedding_index",
#             "path": "embedding",
#             "queryVector": query_vec,
#             "numCandidates": 200,
#             "limit": 5,
#         }
#     },
#     {
#         "$project": {
#             "title": 1,
#             "chunk_index": 1,
#             "text": {"$substr": ["$text", 0, 200]},
#             "score": {"$meta": "vectorSearchScore"},
#             "embedding": 0,
#         }
#     },
# ]
#
# results = list(collection.aggregate(pipeline))
# for r in results:
#     print(f"[{r['score']:.4f}] {r['title']} chunk {r['chunk_index']}")
#     print(f"  {r['text']!r}")
#     print()
print("Atlas Vector Search cell — uncomment when ready.")